In [4]:
import torch
import tiktoken
import gc
from transformers import GPT2Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sd = {k: v.to(device) for k, v in GPT2Model.from_pretrained("gpt2").state_dict().items()}
enc = tiktoken.get_encoding("gpt2")

def layer_norm(x, weight, bias, eps=1e-5):
    mean = x.mean(-1, keepdim=True)
    var = ((x - mean) ** 2).mean(-1, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * weight + bias

def gelu(x):  # GPT-2 tanh approximation
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x ** 3)))

def raw_forward(tokens, ablate_type=None, ablate_layer=None, ablate_head=None, use_sae=False, sae_layer=8):
    n_layers = sae_layer if use_sae else 12
    seq_len = tokens.shape[1]
    n_heads, d_head = 12, 64

    emb = sd['wte.weight'][tokens[0]]              # [seq, 768]
    pos = sd['wpe.weight'][:seq_len]               # [seq, 768]
    x = (emb + pos).unsqueeze(0)                   # [1, seq, 768]

    for layer in range(n_layers):
        # Attention
        ln1_out = layer_norm(x, sd[f'h.{layer}.ln_1.weight'], sd[f'h.{layer}.ln_1.bias'])

        W_qkv = sd[f'h.{layer}.attn.c_attn.weight']  # [768, 2304]
        b_qkv = sd[f'h.{layer}.attn.c_attn.bias']

        q = ln1_out @ W_qkv[:, :768]    + b_qkv[:768]
        k = ln1_out @ W_qkv[:, 768:1536] + b_qkv[768:1536]
        v = ln1_out @ W_qkv[:, 1536:]   + b_qkv[1536:]

        q = q.view(1, seq_len, n_heads, d_head).transpose(1, 2)
        k = k.view(1, seq_len, n_heads, d_head).transpose(1, 2)
        v = v.view(1, seq_len, n_heads, d_head).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (d_head ** 0.5)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        scores.masked_fill_(mask, float('-inf'))
        pattern = torch.softmax(scores, dim=-1)

        z = pattern @ v

        if ablate_type == 'head' and ablate_layer == layer:
            z[0, ablate_head, :, :] = 0

        z = z.transpose(1, 2).contiguous().view(1, seq_len, 768)

        W_proj = sd[f'h.{layer}.attn.c_proj.weight']
        b_proj = sd[f'h.{layer}.attn.c_proj.bias']
        x = x + z @ W_proj + b_proj

        # MLP
        ln2_out = layer_norm(x, sd[f'h.{layer}.ln_2.weight'], sd[f'h.{layer}.ln_2.bias'])

        hidden = gelu(ln2_out @ sd[f'h.{layer}.mlp.c_fc.weight'] + sd[f'h.{layer}.mlp.c_fc.bias'])
        mlp_out = hidden @ sd[f'h.{layer}.mlp.c_proj.weight'] + sd[f'h.{layer}.mlp.c_proj.bias']

        if ablate_type == 'mlp' and ablate_layer == layer:
            mlp_out = torch.zeros_like(mlp_out)

        x = x + mlp_out

    if use_sae:
        return x  # [1, seq, 768] residual stream at sae_layer

    x = layer_norm(x, sd['ln_f.weight'], sd['ln_f.bias'])
    logits = x @ sd['wte.weight'].T  # [1, seq, 50257]
    return logits

gc.collect()
print(f"GPT-2 loaded on {device}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 loaded on cuda


In [5]:
prompt = "The Golden Gate Bridge spans"
token_ids = [50256] + enc.encode(prompt)
generated = list(token_ids)

with torch.no_grad():
    for _ in range(20):
        logits = raw_forward(torch.tensor([generated], device=device))
        generated.append(logits[0, -1].argmax().item())

print(enc.decode(generated))

<|endoftext|>The Golden Gate Bridge spans the San Francisco Bay Area and spans the Bay Area's Pacific Coast. The bridge is the world's


In [ ]:
import torch.nn.functional as F

d_model = 768
d_sae = 4096
l1_coeff = 2
lr = 3e-4
n_steps = 20000
batch_size = 64

training_prompts = [
    "The Golden Gate Bridge spans the San Francisco Bay connecting the city to Marin County",
    "Machine learning models learn patterns from data by adjusting weights through backpropagation",
    "The capital of France is Paris which is known for the Eiffel Tower and the Louvre Museum",
    "Scientists discovered that the universe is expanding at an accelerating rate due to dark energy",
    "Python is a popular programming language used for web development and artificial intelligence",
    "The history of ancient Rome spans over a thousand years from kingdom to republic to empire",
    "Quantum computers use qubits that can exist in superposition of states unlike classical bits",
    "The Amazon rainforest produces roughly twenty percent of the world oxygen supply each year",
    "Neural networks consist of layers of neurons connected by weights that are learned during training",
    "Shakespeare wrote thirty seven plays including Hamlet Macbeth and Romeo and Juliet among others",
    "The stock market crashed in nineteen twenty nine leading to the Great Depression across the world",
    "Photosynthesis converts sunlight water and carbon dioxide into glucose and oxygen in plant cells",
    "The internet was originally developed by DARPA as a military communication network in the sixties",
    "Einstein published his theory of general relativity in nineteen fifteen changing physics forever",
    "DNA contains the genetic instructions for the development and functioning of all living organisms",
    "The moon orbits the Earth approximately every twenty seven days and causes ocean tides on Earth",
    "Artificial intelligence has made significant progress in natural language processing and computer vision",
    "The Great Wall of China stretches over thirteen thousand miles across northern China border regions",
    "Economists study how societies allocate scarce resources among competing wants and unlimited needs",
    "The human brain contains approximately eighty six billion neurons connected by trillions of synapses",
    "Climate change is driven primarily by greenhouse gas emissions from burning fossil fuels worldwide",
    "Mathematics is the foundation of physics engineering computer science and many other scientific fields",
    "The Renaissance began in Italy in the fourteenth century and spread throughout Europe over two centuries",
    "Bacteria are single celled organisms that can be found in virtually every environment on Earth",
    "The periodic table organizes chemical elements by atomic number and electron configuration patterns",
    "Democracy originated in ancient Athens where citizens could vote directly on laws and policy decisions",
    "Gravity is the force that attracts objects with mass toward each other as described by Newton and Einstein",
    "The printing press invented by Gutenberg around fourteen fifty revolutionized the spread of information",
    "Vaccines work by training the immune system to recognize and fight specific pathogens before infection",
    "The speed of light in vacuum is approximately three hundred thousand kilometers per second a universal constant",
    "Cooking food with heat breaks down proteins and starches making nutrients more accessible to the body",
    "The Pacific Ocean is the largest and deepest ocean covering more area than all land masses combined",
    "Electricity flows through conductors when a voltage difference creates an electric field pushing electrons",
    "Music activates multiple brain regions simultaneously including areas for emotion memory and motor control",
    "Volcanic eruptions release magma from beneath the Earth crust along with gases ash and pyroclastic flows",
    "The industrial revolution transformed manufacturing from hand production to machine based factory systems",
    "Black holes form when massive stars collapse under their own gravity creating infinite density singularities",
    "Language acquisition in children follows predictable stages from babbling to single words to full sentences",
    "Tectonic plates float on the mantle and their movements cause earthquakes volcanoes and mountain formation",
    "The human genome contains approximately three billion base pairs encoding roughly twenty thousand genes total",
    "Water molecules consist of two hydrogen atoms bonded to one oxygen atom through covalent bonds",
    "The Roman Empire fell in four seventy six when Germanic tribes overthrew the last Western emperor",
    "Antibiotics kill bacteria by disrupting cell wall synthesis or blocking protein production in ribosomes",
    "The Pythagorean theorem states that in a right triangle the square of the hypotenuse equals the sum of squares",
    "Glaciers form when accumulated snow compresses into dense ice and begins flowing under its own weight",
    "The French Revolution began in seventeen eighty nine with the storming of the Bastille prison in Paris",
    "Mitochondria generate adenosine triphosphate through oxidative phosphorylation in the electron transport chain",
    "The speed of sound in air at sea level is approximately three hundred forty three meters per second",
    "Coral reefs support roughly twenty five percent of all marine species despite covering less than one percent",
    "The Wright brothers achieved the first powered flight at Kitty Hawk in nineteen oh three lasting twelve seconds",
    "Semiconductors conduct electricity between conductors and insulators making transistors and microchips possible",
    "The Sahara Desert spans over nine million square kilometers across eleven countries in northern Africa",
    "Plate tectonics explains how continents drift apart and collide over millions of years reshaping Earth surface",
    "The Hubble Space Telescope orbits Earth at five hundred forty kilometers capturing deep field images of galaxies",
    "Proteins fold into three dimensional shapes determined by their amino acid sequence and chemical environment",
    "The Treaty of Versailles signed in nineteen nineteen imposed heavy reparations on Germany after World War One",
    "Superconductors carry electrical current with zero resistance when cooled below their critical temperature",
    "The Amazon River carries more water than any other river discharging two hundred thousand cubic meters per second",
    "Neurotransmitters cross the synaptic cleft to bind receptors on the postsynaptic neuron triggering signals",
    "The Magna Carta signed in twelve fifteen established that the king was subject to the rule of law",
    "Solar panels convert photons into electrons through the photovoltaic effect in silicon semiconductor junctions",
    "Mount Everest stands at eight thousand eight hundred forty eight meters as the tallest peak above sea level",
    "The double helix structure of DNA was discovered by Watson and Crick in nineteen fifty three using X ray data",
    "Entropy in thermodynamics measures the disorder of a system and always increases in isolated systems over time",
    "The Silk Road connected China to the Mediterranean facilitating trade in silk spices and ideas for centuries",
    "CRISPR allows precise editing of DNA sequences by using guide RNA to direct the Cas9 enzyme to cut specific genes",
    "The Mariana Trench reaches nearly eleven thousand meters deep making it the deepest point in Earth oceans",
    "Inflation in cosmology refers to the rapid exponential expansion of space in the first fraction of a second",
    "The Ottoman Empire lasted over six hundred years from twelve ninety nine to nineteen twenty two spanning three continents",
    "Ribosomes translate messenger RNA into proteins by reading codons and assembling amino acids in sequence",
    "The Doppler effect causes the frequency of waves to increase as the source approaches and decrease as it recedes",
    "Antarctica contains roughly seventy percent of all fresh water on Earth locked in its ice sheet",
    "The Manhattan Project developed the first nuclear weapons during World War Two at Los Alamos laboratory",
    "Catalysts speed up chemical reactions by lowering activation energy without being consumed in the process",
    "The Fibonacci sequence appears throughout nature in sunflower spirals pinecone patterns and nautilus shells",
    "The Suez Canal connects the Mediterranean Sea to the Red Sea eliminating the need to sail around Africa",
    "Stem cells can differentiate into specialized cell types making them valuable for regenerative medicine research",
    "The Big Bang theory describes the universe expanding from an extremely hot dense state fourteen billion years ago",
    "Archimedes principle states that buoyant force equals the weight of fluid displaced by a submerged object",
    "The Trans Siberian Railway stretches over nine thousand kilometers from Moscow to Vladivostok across Russia",
    "Hormones are chemical messengers produced by endocrine glands that regulate metabolism growth and reproduction",
    "General relativity predicts that massive objects curve spacetime causing light to bend around stars and galaxies",
    "The Panama Canal uses a system of locks to raise and lower ships eighty five feet between ocean levels",
    "Chlorophyll absorbs red and blue light while reflecting green giving plants their characteristic color",
    "The Cold War lasted from nineteen forty seven to nineteen ninety one between the United States and Soviet Union",
    "Enzymes are biological catalysts that lower activation energy for specific biochemical reactions in cells",
    "The circumference of Earth at the equator is approximately forty thousand seventy five kilometers",
    "Symbiosis describes close biological interactions between species including mutualism parasitism and commensalism",
    "The Apollo eleven mission landed humans on the Moon on July twentieth nineteen sixty nine for the first time",
    "Electromagnetic waves travel at the speed of light and include radio microwaves infrared visible ultraviolet X rays",
    "The Nile River flows north for over six thousand six hundred kilometers through eleven African countries",
    "Chromosomes are thread like structures of coiled DNA found in the nucleus that carry genetic information",
    "Supply and demand determine prices in free markets where equilibrium occurs at the intersection of both curves",
    "The speed of Earth rotation at the equator is approximately one thousand six hundred seventy kilometers per hour",
    "Osmosis is the movement of water molecules across a semipermeable membrane from low to high solute concentration",
    "The Library of Alexandria was one of the largest and most significant libraries of the ancient world",
    "Nuclear fusion powers the Sun by combining hydrogen nuclei into helium releasing enormous amounts of energy",
    "The human heart beats approximately one hundred thousand times per day pumping about seven thousand liters of blood",
    "Natural selection drives evolution by favoring organisms with traits better suited to their environment",
    "The Great Barrier Reef stretches over two thousand three hundred kilometers along the northeast coast of Australia",
    "Magnetism arises from the motion of electric charges and the intrinsic magnetic moments of elementary particles",
    "The Rosetta Stone discovered in seventeen ninety nine enabled scholars to decode Egyptian hieroglyphics",
    "Photons are massless particles that carry electromagnetic force and exhibit both wave and particle properties",
    "The Mississippi River flows two thousand three hundred miles from Minnesota to the Gulf of Mexico",
    "Cellular respiration converts glucose and oxygen into carbon dioxide water and adenosine triphosphate energy",
    "The theory of evolution by natural selection was independently proposed by Darwin and Wallace in eighteen fifty eight",
    "Transistors are semiconductor devices that amplify or switch electronic signals forming the basis of modern computers",
    "The Indian Ocean covers approximately seventy million square kilometers making it the third largest ocean",
    "Atmospheric pressure at sea level equals approximately one hundred one thousand three hundred twenty five pascals",
    "The Gutenberg Bible printed around fourteen fifty five was the first major book produced with movable type",
    "Mitosis is the process of cell division that produces two genetically identical daughter cells from one parent",
    "The velocity of an object in orbit depends on its altitude with lower orbits requiring higher speeds",
    "Penicillin discovered by Alexander Fleming in nineteen twenty eight was the first widely used antibiotic medicine",
    "The wavelength of visible light ranges from about three hundred eighty to seven hundred nanometers",
    "Continental drift was proposed by Alfred Wegener in nineteen twelve based on matching coastlines and fossils",
    "The human eye can distinguish approximately ten million different colors using three types of cone cells",
    "Radioactive decay occurs when unstable atomic nuclei lose energy by emitting radiation in the form of particles",
    "The Mediterranean Sea is connected to the Atlantic Ocean through the narrow Strait of Gibraltar",
    "Algorithms are step by step procedures for solving problems or performing computations in a finite number of steps",
    "The Andes mountain range extends over seven thousand kilometers along the western coast of South America",
    "Insulin regulates blood sugar levels by signaling cells to absorb glucose from the bloodstream after meals",
    "The electromagnetic spectrum spans from long radio waves to short gamma rays with visible light in between",
    "Coral bleaching occurs when rising ocean temperatures cause corals to expel the symbiotic algae living in them",
    "The first programmable computer was built by Konrad Zuse in nineteen forty one using electromechanical relays",
    "Sound waves are longitudinal pressure waves that propagate through a medium by compression and rarefaction",
    "The Black Death killed roughly one third of Europe population between thirteen forty seven and thirteen fifty one",
    "Quantum entanglement links particles so that measuring one instantly determines the state of the other regardless of distance",
    "The Gulf Stream carries warm water from the Gulf of Mexico across the Atlantic moderating European climate",
    "Carbon dating measures the ratio of carbon fourteen to carbon twelve to determine the age of organic materials",
    "The Bermuda Triangle is a loosely defined region in the North Atlantic where ships and aircraft have disappeared",
    "Recombinant DNA technology allows scientists to combine genetic material from different organisms for research",
    "The Aurora Borealis occurs when charged particles from the Sun interact with gases in Earth magnetic field",
    "Brownian motion describes the random movement of particles suspended in fluid caused by molecular collisions",
    "The Dead Sea sits at four hundred thirty meters below sea level making it the lowest point on land",
    "Gene expression is the process by which information from a gene directs the synthesis of a functional product",
    "The Richter scale measures earthquake magnitude logarithmically so each whole number is ten times more amplitude",
    "Nitrogen makes up approximately seventy eight percent of Earth atmosphere by volume with oxygen at twenty one",
    "The printing of money without backing increases inflation by reducing the purchasing power of each currency unit",
    "Diffraction occurs when waves encounter obstacles or openings causing them to bend and spread into shadow regions",
    "The human skeleton contains two hundred six bones that provide structure protection and facilitate movement",
    "Plate boundaries are classified as convergent divergent or transform based on relative motion between plates",
    "The Voyager one spacecraft launched in nineteen seventy seven is the most distant human made object from Earth",
    "Aerobic exercise strengthens the cardiovascular system by increasing heart rate and oxygen consumption during activity",
    "The half life of a radioactive isotope is the time required for half of its atoms to decay into other elements",
    "Photosynthetic organisms produce oxygen as a byproduct splitting water molecules using energy from sunlight",
    "The Coriolis effect causes moving objects on Earth to deflect right in the northern hemisphere and left in the south",
    "Alloys are mixtures of metals that combine desirable properties like strength and corrosion resistance of each component",
    "The International Space Station orbits Earth at approximately four hundred kilometers altitude completing one orbit every ninety minutes",
]

all_acts = []
with torch.no_grad():
    for i, p in enumerate(training_prompts):
        tokens = torch.tensor([[50256] + enc.encode(p)], device=device)
        resid = raw_forward(tokens, use_sae=True)
        all_acts.append(resid[0].clone())
        del resid, tokens
        if i % 10 == 0:
            gc.collect()

all_acts = torch.cat(all_acts, dim=0)  # [total_tokens, 768]
gc.collect()

W_enc_t = (torch.randn(d_model, d_sae, device=device) * (1 / d_model ** 0.5)).requires_grad_(True)
b_enc_t = torch.zeros(d_sae, device=device).requires_grad_(True)
W_dec_t = F.normalize(torch.randn(d_sae, d_model, device=device), dim=-1).requires_grad_(True)
b_dec_t = all_acts.mean(dim=0).clone().requires_grad_(True)

optimizer = torch.optim.Adam([W_enc_t, b_enc_t, W_dec_t, b_dec_t], lr=lr)

# Loss = ||x - x_hat||² + λ||features||₁
# encode: features = ReLU((x - b_dec) @ W_enc + b_enc)
# decode: x_hat = features @ W_dec + b_dec
for step in range(n_steps):
    idx = torch.randint(0, all_acts.shape[0], (batch_size,))
    x = all_acts[idx]

    features = torch.relu((x - b_dec_t) @ W_enc_t + b_enc_t)
    x_hat = features @ W_dec_t + b_dec_t

    recon_loss = (x - x_hat).pow(2).mean()
    l1_loss = features.abs().mean()
    loss = recon_loss + l1_coeff * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        W_dec_t.data = F.normalize(W_dec_t.data, dim=-1)

print(f"SAE done. recon={recon_loss.item():.4f}, active={(features > 0).float().mean().item():.1%}")

SAE done. recon=0.0394, active=3.9%
